# HealthMind NLP — Test Suite
Tests `nlp_engine.analyze_message` and `feeling_engine.analyze_feeling`.

Run cells top to bottom. Models load once in Cell 2 and are reused for all tests.

## Cell 2 — Load both engines (models download on first run, ~1-2 min)

In [3]:
# pyright: reportMissingImports=false
import sys
from pathlib import Path

# ── Point this to the real NLP package root ────────────────────────────────
NLP_DIR = Path("..").resolve()   # project root (the folder that contains chatbot/)
sys.path.insert(0, str(NLP_DIR))

from lumina.nlp.nlp_engine import analyze_message  # type: ignore[reportMissingImports]
from lumina.nlp.feeling_engine import analyze_feeling  # type: ignore[reportMissingImports]

print("✅ Both engines loaded successfully.")

✅ Both engines loaded successfully.


## Cell 3 — Pretty-print helper

In [4]:
import json

def show(label: str, result: dict):
    print(f"\n{'='*60}")
    print(f"  {label}")
    print('='*60)
    print(json.dumps(result, indent=2, ensure_ascii=False))

print("✅ Helper ready.")

✅ Helper ready.


---
## Cell 4 — Test: normal distress message
Expected: negative sentiment, sadness/fear emotion, anxiety/depression pain points, venting or seeking_advice intent, no crisis flag.

In [5]:
msg = "I've been feeling really anxious and can't sleep. My mind won't stop racing and I feel completely alone."

show("NLP — normal distress",     analyze_message(msg))
show("FEELING — normal distress", analyze_feeling(msg))


  NLP — normal distress
{
  "cleaned_text": "I've been feeling really anxious and can't sleep. My mind won't stop racing and I feel completely alone.",
  "tokens": [
    "anxious",
    "sleep",
    "mind",
    "stop",
    "race",
    "completely"
  ],
  "entities": {
    "SYMPTOM": [
      "can't sleep"
    ],
    "DISORDER": [],
    "MEDICATION": [],
    "THERAPY": [],
    "CRISIS_SIGNAL": [],
    "EMOTION_STATE": [
      "anxious"
    ],
    "COPING_STRATEGY": [],
    "SUPPORT_RESOURCE": []
  },
  "pain_points": [
    "sleep problems and exhaustion",
    "loneliness and social isolation",
    "depression and hopelessness"
  ],
  "primary_macro": "emotional distress",
  "intent": "venting",
  "intent_confidence": 1.0,
  "intent_ambiguous": null,
  "crisis_flag": false
}

  FEELING — normal distress
{
  "sentiment": "negative",
  "sentiment_score": 0.9259,
  "sentiment_scores": {
    "negative": 0.9259,
    "neutral": 0.0678,
    "positive": 0.0064
  },
  "emotion": "fear",
  "emotion

helped me add these to the symptom patterns:
"can't sleep", "cannot sleep", "trouble sleeping", 
"trouble falling asleep", "can't fall asleep",

---
## Cell 5 — Test: explicit crisis message
**Critical test.** `crisis_flag` MUST be `True`. Intent MUST be `"crisis"`.
If either is wrong, stop and fix before going further.

In [8]:
msg = "I can't take this anymore. I've been thinking about killing myself every day this week."

# Check what clean_text produces
from lumina.nlp.nlp_engine import clean_text, detect_crisis

cleaned = clean_text(msg)
print(f"Cleaned : '{cleaned}'")
print(f"Crisis  : {detect_crisis(cleaned)}")

# Check which keywords are being tested against
from lumina.nlp.lexicons import CRISIS_KEYWORDS
print(f"\nCrisis keywords: {CRISIS_KEYWORDS}")

# Manual check — does the phrase appear in cleaned text?
for kw in CRISIS_KEYWORDS:
    if kw in cleaned.lower():
        print(f"  MATCH: '{kw}'")

Cleaned : 'I can't take this anymore. I've been thinking about killing myself every day this week.'
Crisis  : True

Crisis keywords: {'ending it all', 'thinking about killing', 'better off dead', 'no will to live', 'thinking about suicide', 'ending my life', 'not want to live', 'wanting to die', 'take my life', 'cut myself', 'self harm', 'od on', "don't want to live", 'hurt myself', 'end it all', 'hanging myself', 'thought about suicide', 'dont want to live', "don't want to be here", 'self-harm', 'hurting myself', 'killing myself', 'cutting myself', 'overdose', 'dont want to be here', 'suicide', 'take my own life', 'wish i were dead', 'suicidal', 'want to die', 'no reason to live', 'nervous breakdown', 'kill myself', 'end my life', 'wish i was dead'}
  MATCH: 'thinking about killing'
  MATCH: 'killing myself'


helped add more crisis words. might have to look into a way to try and lemmatize in a way words like that to reduce the amount of words and dictionaries in the nlp_engine file

In [4]:
msg = "I can't take this anymore. I've been thinking about killing myself every day this week."

nlp_result = analyze_message(msg)
show("NLP — crisis message", nlp_result)

# Hard assertion — this must never fail
assert nlp_result["crisis_flag"] is True,  "❌ CRITICAL: crisis_flag should be True"
assert nlp_result["intent"] == "crisis",   "❌ CRITICAL: intent sho uld be 'crisis'"
print("\n✅ Crisis flag and intent are correct.")


  NLP — crisis message
{
  "cleaned_text": "I can't take this anymore. I've been thinking about killing myself every day this week.",
  "tokens": [
    "anymore",
    "kill",
    "week"
  ],
  "entities": {
    "SYMPTOM": [],
    "DISORDER": [],
    "MEDICATION": [],
    "THERAPY": [],
    "CRISIS_SIGNAL": [
      "thinking about killing"
    ],
    "EMOTION_STATE": [],
    "COPING_STRATEGY": [],
    "SUPPORT_RESOURCE": []
  },
  "pain_points": [
    "suicidal thoughts and self-harm",
    "depression and hopelessness",
    "physical symptoms of stress"
  ],
  "primary_macro": "trauma & crisis",
  "intent": "crisis",
  "intent_confidence": 1.0,
  "intent_ambiguous": null,
  "crisis_flag": true
}

✅ Crisis flag and intent are correct.


---
## Cell 6 — Test: false-positive crisis check
Known edge case from the code review: `"want to die"` inside an innocent phrase.
This WILL currently trip the crisis flag — cell documents the known tradeoff.

In [9]:
msg = "That concert was so good I want to die laughing, it was unreal."

nlp_result = analyze_message(msg)
print(f"crisis_flag : {nlp_result['crisis_flag']}")
print(f"intent      : {nlp_result['intent']}")

if nlp_result["crisis_flag"]:
    print("\n⚠️  False positive: substring match tripped the crisis flag.")
    print("   This is a known, accepted tradeoff (safe > sorry).")
    print("   Document it and make sure empathy_engine handles it gracefully.")
else:
    print("\n✅ No false positive on this phrase.")

crisis_flag : True
intent      : crisis

⚠️  False positive: substring match tripped the crisis flag.
   This is a known, accepted tradeoff (safe > sorry).
   Document it and make sure empathy_engine handles it gracefully.


false flag indeed as it flags die as a crisis where its used so we have to find a way to work around this in the empathy file

nlp_engine      → crisis_flag: True
feeling_engine  → sentiment: positive, emotion: joy
                         ↓
empathy_engine  → combines both → decides: real crisis or false positive?

---
## Cell 7 — Test: question detection edge case
Known weak point: questions not ending with `?` won't set `has_question`.
Watch `intent` — it may land on `venting` instead of `seeking_advice`.

In [7]:
explicit_q   = "Should I start taking antidepressants for my depression?"
implicit_q   = "I don't know if I should start taking antidepressants for my depression"

r1 = analyze_message(explicit_q)
r2 = analyze_message(implicit_q)

print(f"Explicit '?' → intent: {r1['intent']} (confidence: {r1['intent_confidence']})")
print(f"Implicit Q   → intent: {r2['intent']} (confidence: {r2['intent_confidence']})")

if r1["intent"] != r2["intent"]:
    print("\n⚠️  Same semantic question classified differently due to '?' detection.")
    print("   Consider scanning full text for question words (why/how/should/what).")

Explicit '?' → intent: seeking_advice (confidence: 0.71)
Implicit Q   → intent: seeking_advice (confidence: 0.57)


so far so good

---
## Cell 8 — Test: entity extraction
Checks that medication, disorder, and therapy terms are correctly picked up.

In [8]:
msg = "I was diagnosed with PTSD and my psychiatrist put me on sertraline. I've also been doing CBT for about three months."

result = analyze_message(msg)
entities = result["entities"]

print("Extracted entities:")
for label, values in entities.items():
    if values:
        print(f"  {label:20s}: {values}")

# Soft assertions
assert "ptsd" in entities.get("DISORDER", []),    "❌ PTSD not found in DISORDER"
assert "sertraline" in entities.get("MEDICATION", []), "❌ sertraline not found in MEDICATION"
assert "cbt" in entities.get("THERAPY", []),      "❌ CBT not found in THERAPY"
print("\n✅ All expected entities extracted.")

Extracted entities:
  DISORDER            : ['ptsd']
  MEDICATION          : ['sertraline']
  THERAPY             : ['cbt']
  SUPPORT_RESOURCE    : ['psychiatrist']

✅ All expected entities extracted.


yup

---
## Cell 9 — Test: emotion ambiguity flag
Short or emotionally neutral text should trigger `emotion_ambiguous = True`.

In [9]:
clear_msg = "I feel completely hopeless and I've been crying for three days straight. I don't see any way out."
vague_msg = "I don't know, things are just different lately."

r_clear = analyze_feeling(clear_msg)
r_vague = analyze_feeling(vague_msg)

print(f"Clear distress → emotion: {r_clear['emotion']:12s} | ambiguous: {r_clear['emotion_ambiguous']} | score: {r_clear['emotion_score']}")
print(f"Vague message  → emotion: {r_vague['emotion']:12s} | ambiguous: {r_vague['emotion_ambiguous']} | score: {r_vague['emotion_score']}")

Clear distress → emotion: sadness      | ambiguous: False | score: 0.9799
Vague message  → emotion: neutral      | ambiguous: False | score: 0.7065


The vague message correctly landed on neutral, but ambiguous: False at 0.7065 is worth flagging. 

---
## Cell 10 — Test: sentiment vs emotion mismatch
The code comments mention this case. Surface it explicitly so empathy_engine can handle it.

In [10]:
# Sarcasm / dark humor — sentiment may read positive while emotion is negative
msg = "Oh great, another panic attack at 3am. Love this for me."

result = analyze_feeling(msg)
print(f"Sentiment : {result['sentiment']} ({result['sentiment_score']})")
print(f"Emotion   : {result['emotion']} ({result['emotion_score']})")

if result["sentiment"] == "positive" and result["emotion"] in ("fear", "sadness", "anger"):
    print("\n⚠️  Mismatch detected: positive sentiment but negative emotion.")
    print("   empathy_engine should trust emotion over sentiment in this case.")

Sentiment : positive (0.4556)
Emotion   : fear (0.9839)

⚠️  Mismatch detected: positive sentiment but negative emotion.
   empathy_engine should trust emotion over sentiment in this case.


the mismatch detector is working exactly as intended.

---
## Cell 11 — Test: positive / neutral message
Makes sure the engine doesn't over-pathologize normal input.

In [11]:
msg = "I had a really good therapy session today. I feel like things are finally starting to click."

nlp     = analyze_message(msg)
feeling = analyze_feeling(msg)

print(f"crisis_flag : {nlp['crisis_flag']}")
print(f"intent      : {nlp['intent']}")
print(f"sentiment   : {feeling['sentiment']}")
print(f"emotion     : {feeling['emotion']}")

assert nlp["crisis_flag"] is False, "❌ False positive crisis on positive message"
print("\n✅ Positive message handled correctly.")

crisis_flag : False
intent      : venting
sentiment   : positive
emotion     : joy

✅ Positive message handled correctly.


Perfect

---
## Cell 12 — Batch smoke test
Runs 8 varied messages and prints a summary table. Good for spotting systemic issues.

In [12]:
test_cases = [
    ("crisis",    "I want to kill myself. I have a plan."),
    ("distress",  "I've been so depressed lately, I can barely get out of bed."),
    ("advice",    "Should I try DBT or CBT for my borderline personality disorder?"),
    ("positive",  "My anxiety has been so much better since I started meditation."),
    ("venting",   "My boss doesn't care about my burnout at all and I'm exhausted."),
    ("neutral",   "I'm thinking about looking into online therapy options."),
    ("physical",  "My heart races and my chest gets tight when I have to present at work."),
    ("ambiguous", "I don't know. Everything just feels off."),
]

print(f"{'Label':<12} {'Crisis':>7} {'Intent':<22} {'Sentiment':<11} {'Emotion':<12} {'Ambig':>6}")
print("-" * 78)

for label, msg in test_cases:
    nlp     = analyze_message(msg)
    feeling = analyze_feeling(msg)
    print(
        f"{label:<12} "
        f"{str(nlp['crisis_flag']):>7} "
        f"{nlp['intent']:<22} "
        f"{feeling['sentiment']:<11} "
        f"{feeling['emotion']:<12} "
        f"{str(feeling['emotion_ambiguous']):>6}"
    )

Label         Crisis Intent                 Sentiment   Emotion       Ambig
------------------------------------------------------------------------------
crisis          True crisis                 negative    neutral        True
distress       False venting                negative    sadness       False
advice         False seeking_advice         neutral     fear          False
positive       False venting                positive    fear          False
venting        False venting                negative    neutral        True
neutral        False venting                neutral     joy           False
physical       False sharing_experience     negative    fear          False
ambiguous      False venting                negative    neutral       False


3 things to fix in the existing files:


1. help_seeking intent never fires : The neutral row ("I'm thinking about looking into online therapy options") landed on venting when it should be help_seeking. The intent label is too narrow.

2. Both positive and advice rows got emotion: fear despite neutral/positive framing. The model latched onto the words "anxiety" and "borderline personality disorder"

3. The ambiguous row ("Everything just feels off") returned ambiguous: False at 0.70. That message is genuinely uncertain. 

2 things to handle later in empathy_engine.py:

1. Crisis false positives
crisis_flag can trip on innocent phrases. Don't fix in nlp_engine — handle in empathy_engine

2. Sentiment/emotion mismatch on sarcasm When emotion is high confidence negative but sentiment reads positive, trust the emotion(Right now feeling_engine just reports the data honestly. But someone has to actually act on that mismatch, and that's empathy_engine's job.)

---
## Cell 13 — Load empathy_engine
Must be run after Cell 2 (both engines already loaded). Imports `build_prompt` which orchestrates everything.


In [ ]:
from lumina.dialogue.empathy_engine import build_prompt  # type: ignore[reportMissingImports]
print("✅ empathy_engine loaded.")

---
## Cell 14 — Test: venting + sadness
Expected: warm/validating strategy, no advice instruction, negative sentiment resolved correctly.


In [ ]:
msg = "I've been feeling really anxious and can't sleep. My mind won't stop racing and I feel completely alone."

result = build_prompt(msg)

print(f"Intent          : {result['intent']}")
print(f"Strategy        : {result['strategy']}")
print(f"Crisis confirmed: {result['crisis_confirmed']}")
print(f"RAG hits        : {result['rag_hits']}")
print()
print("=" * 60)
print("SYSTEM PROMPT:")
print("=" * 60)
print(result['system_prompt'])

---
## Cell 15 — Test: crisis message (full pipeline)
**Critical.** `crisis_confirmed` MUST be `True`. Crisis protocol block must appear in the system prompt.


In [ ]:
msg = "I can't take this anymore. I've been thinking about killing myself every day this week."

result = build_prompt(msg)

assert result['crisis_confirmed'] is True, "❌ CRITICAL: crisis_confirmed should be True"
assert result['intent'] == 'crisis',        "❌ CRITICAL: intent should be 'crisis'"
assert '988' in result['system_prompt'],    "❌ CRITICAL: 988 crisis line missing from prompt"

print(f"Intent          : {result['intent']}")
print(f"Crisis confirmed: {result['crisis_confirmed']}")
print(f"RAG hits        : {result['rag_hits']} (should be 0 — RAG suppressed in crisis)")
print()
print("=" * 60)
print("SYSTEM PROMPT:")
print("=" * 60)
print(result['system_prompt'])
print()
print("✅ Crisis pipeline correct.")

---
## Cell 16 — Test: false positive crisis resolved by empathy_engine
The nlp_engine flags `"want to die laughing"` as crisis. empathy_engine should resolve it using `feeling_engine` output (positive sentiment + joy emotion).


In [ ]:
msg = "That concert was so good I want to die laughing, it was unreal."

result = build_prompt(msg)

print(f"Intent          : {result['intent']} (should NOT be 'crisis')")
print(f"Crisis confirmed: {result['crisis_confirmed']} (should be False)")
print(f"Strategy        : {result['strategy']}")
print()
debug_nlp     = result['debug']['nlp']
debug_feeling = result['debug']['feeling']
print(f"nlp crisis_flag : {debug_nlp['crisis_flag']} (True — keyword matched)")
print(f"feeling emotion : {debug_feeling['emotion']} | sentiment: {debug_feeling['sentiment']}")
print()
if not result['crisis_confirmed']:
    print("✅ False positive correctly resolved — empathy_engine overrode nlp_engine.")
else:
    print("⚠️  False positive NOT resolved — empathy_engine still escalated to crisis.")
    print("   Check _resolve_intent() in empathy_engine.py.")

---
## Cell 17 — Test: sarcasm / dark humor mismatch
Emotion should be high-confidence negative (fear/anger). Sentiment may read positive.
empathy_engine should trust emotion over sentiment and NOT pick a positive strategy.


In [ ]:
msg = "Oh great, another panic attack at 3am. Love this for me."

result = build_prompt(msg)

debug_feeling = result['debug']['feeling']
print(f"Sentiment       : {debug_feeling['sentiment']} (score: {debug_feeling['sentiment_score']})")
print(f"Emotion         : {debug_feeling['emotion']} (score: {debug_feeling['emotion_score']})")
print(f"Intent          : {result['intent']}")
print(f"Strategy        : {result['strategy']}")
print()
if debug_feeling['sentiment'] == 'positive' and debug_feeling['emotion'] in ('fear', 'sadness', 'anger'):
    print("⚠️  Mismatch detected — empathy_engine resolved it.")
    if 'positive' not in result['strategy'].lower() and 'casual' not in result['strategy'].lower():
        print("✅ Strategy is correctly negative-leaning despite positive sentiment.")
    else:
        print("❌ Strategy is still positive — mismatch not handled.")
else:
    print("No sentiment/emotion mismatch on this message.")

---
## Cell 18 — Test: entity injection into prompt
Checks that detected entities (disorder, medication, therapy) actually appear in the system prompt context block.


In [ ]:
msg = "I was diagnosed with PTSD and my psychiatrist put me on sertraline. I've also been doing CBT for about three months."

result = build_prompt(msg)
prompt = result['system_prompt']

entities = result['debug']['nlp']['entities']
print("Detected entities:")
for label, values in entities.items():
    if values:
        print(f"  {label:20s}: {values}")
print()
print(f"RAG hits: {result['rag_hits']} (should be 1-3 for ptsd/sertraline/cbt)")
print()
# Check entity labels appear in the prompt
checks = [
    ('ptsd',       'WHAT THEY MENTIONED'),
    ('sertraline', 'WHAT THEY MENTIONED'),
    ('cbt',        'WHAT THEY MENTIONED'),
]
all_ok = True
for term, section in checks:
    found = term in prompt.lower()
    status = '✅' if found else '❌'
    print(f"{status} '{term}' in prompt: {found}")
    if not found: all_ok = False
if all_ok:
    print("\n✅ All entities injected into system prompt.")

---
## Cell 19 — Test: RAG context injection
Checks that relevant knowledge from `_STATIC_KNOWLEDGE` is injected when entities match, and suppressed during crisis.


In [ ]:
# Should inject RAG — known disorder + therapy
msg_rag    = "I have OCD and I've been doing ERP, but my intrusive thoughts are getting worse."
# Should suppress RAG — crisis overrides knowledge injection
msg_crisis = "I want to end my life tonight."

r_rag    = build_prompt(msg_rag)
r_crisis = build_prompt(msg_crisis)

print(f"Normal message  — RAG hits: {r_rag['rag_hits']}   (expected > 0)")
print(f"Crisis message  — RAG hits: {r_crisis['rag_hits']} (expected 0 — suppressed)")
print()
rag_in_normal = 'RELEVANT CONTEXT' in r_rag['system_prompt']
rag_in_crisis = 'RELEVANT CONTEXT' in r_crisis['system_prompt']
print(f"'RELEVANT CONTEXT' in normal prompt : {rag_in_normal} (should be True)")
print(f"'RELEVANT CONTEXT' in crisis prompt : {rag_in_crisis} (should be False)")
if rag_in_normal and not rag_in_crisis:
    print("\n✅ RAG injected for normal, suppressed for crisis.")

---
## Cell 20 — Test: positive message not over-pathologized
The engine should not assign a heavy crisis/distress strategy to a clearly positive message.


In [ ]:
msg = "I had a really good therapy session today. I feel like things are finally starting to click."

result = build_prompt(msg)

print(f"Intent          : {result['intent']}")
print(f"Crisis confirmed: {result['crisis_confirmed']} (must be False)")
print(f"Strategy        : {result['strategy']}")
print(f"Sentiment       : {result['debug']['feeling']['sentiment']}")
print(f"Emotion         : {result['debug']['feeling']['emotion']}")

assert result['crisis_confirmed'] is False, "❌ False positive crisis on positive message"
print("\n✅ Positive message not over-pathologized.")

---
## Cell 21 — Batch test: full pipeline summary
Runs all 8 batch messages through `build_prompt` and prints a compact strategy table.
Useful for spotting systemic issues in intent resolution and strategy selection.


In [ ]:
test_cases = [
    ("crisis",    "I want to kill myself. I have a plan."),
    ("distress",  "I've been so depressed lately, I can barely get out of bed."),
    ("advice",    "Should I try DBT or CBT for my borderline personality disorder?"),
    ("positive",  "My anxiety has been so much better since I started meditation."),
    ("venting",   "My boss doesn't care about my burnout at all and I'm exhausted."),
    ("neutral",   "I'm thinking about looking into online therapy options."),
    ("physical",  "My heart races and my chest gets tight when I have to present at work."),
    ("ambiguous", "I don't know. Everything just feels off."),
]

print(f"{'Label':<12} {'Crisis':>7} {'Intent':<22} {'Strategy':<35} {'RAG':>4}")
print("-" * 85)

for label, msg in test_cases:
    r = build_prompt(msg)
    print(
        f"{label:<12} "
        f"{str(r['crisis_confirmed']):>7} "
        f"{r['intent']:<22} "
        f"{r['strategy']:<35} "
        f"{r['rag_hits']:>4}"
    )

---
## Cell 22 — Inspect full system prompt for any message
Use this cell ad-hoc to read the exact prompt that would be sent to the LLM.


In [ ]:
msg = "I've been having intrusive thoughts about hurting people and I'm terrified of myself."

result = build_prompt(msg)

print(f"Intent   : {result['intent']}")
print(f"Strategy : {result['strategy']}")
print(f"Crisis   : {result['crisis_confirmed']}")
print(f"RAG hits : {result['rag_hits']}")
print()
print("=" * 60)
print(result['system_prompt'])
print()
print("=" * 60)
print("DEBUG — NLP:")
import json
print(json.dumps(result['debug']['nlp'],     indent=2))
print()
print("DEBUG — FEELING:")
print(json.dumps(result['debug']['feeling'], indent=2))

---------------------------------
----------------------------------
-----------------------------------

tafsir claude lel empathy file linahki aalih lelmaalouma :

empathy_engine sits between your two engines and the LLM. Here's what it does in plain terms:

It takes the outputs of both engines and decides how the chatbot should respond.
It doesn't analyze anything itself — that's already done. It just reads the results and makes decisions.

Concretely, for every message it:
1. Confirms the intent
Reads crisis_flag, intent, sentiment, emotion together and resolves any conflicts — like the false positive crisis case and the sarcasm mismatch we discussed.
2. Picks a response strategy
Based on intent + emotion it decides the approach:

User is venting + sadness → just validate, don't give advice
User is venting + anger → acknowledge, don't challenge or correct
User is seeking advice → give practical suggestions
User is in crisis → safety first, include resources like 988
User is help seeking → provide therapy/medication information
User is positive → warm and affirming, don't over-therapize

3. Builds a system prompt for the LLM
Translates that strategy into actual instructions for the LLM. Something like:
"The user is venting about work burnout and feels exhausted and angry. 
Do not give advice unless asked. Validate their feelings first. 
Tone: warm, human, not clinical. Avoid suggesting they see a therapist 
immediately — they haven't asked for that."
4. Passes everything to llm_client
It hands off the system prompt + original message to llm_client.py which makes the actual API call. empathy_engine never calls the LLM directly.

A real example using your batch results:
For the venting row — "My boss doesn't care about my burnout at all and I'm exhausted":
crisis_flag : False
intent      : venting
sentiment   : negative
emotion     : neutral  (ambiguous: True)
entities    : SYMPTOM: burnout, EMOTION_STATE: exhausted
pain_points : work stress and burnout
empathy_engine reads this and decides:

No crisis → skip escalation
Venting + ambiguous emotion → be conservative, just listen
burnout and exhausted found → acknowledge those specifically
Don't suggest therapy, don't give advice, don't ask lots of questions

Then builds a prompt like:
"The user is venting about work stress. They feel exhausted and 
unheard by their boss. Emotional signal is ambiguous — respond 
with validation only. Do not offer solutions or suggest professional 
help. Keep the tone human and warm. Reference their burnout directly 
to show you heard them."
That's the whole job — translate engine outputs into LLM instructions.

aya bye

In [1]:
# Cell 23 — LLM client smoke test
import os
import sys
from pathlib import Path

# Ensure the project root is on sys.path for this notebook
NLP_DIR = Path("..").resolve()   # project root (the folder that contains chatbot/)
sys.path.insert(0, str(NLP_DIR))

api_key = os.getenv("MOONSHOT_API_KEY", "")   # was NVIDIA_API_KEY; the app uses Moonshot (chatbot/config.py)
if not api_key:
    print("⚠️  MOONSHOT_API_KEY not set; skipping LLM call.")
else:
    from lumina.dialogue.llm_client import get_response  # type: ignore[reportMissingImports]
    msg = "I have been feeling overwhelmed lately and could use a kind response."
    result = get_response(msg)
    print("LLM response:\n")
    print(result.get("response", result))

c:\Users\Bichr\miniconda3\envs\data_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 990.57it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: cross-encoder/nli-MiniLM2-L6-H768
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1036.50it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-

LLM response:

It sounds like you're carrying a lot right now — the overwhelm, the physical stress, the pressure from work and school. That heaviness is real, and it makes sense that you're feeling this way. I'm really glad you reached out. What's feeling hardest to hold today?
